In [1]:
import numpy as np
import dask as ds
import earthaccess as ea
import xarray as xr
import sys
import pandas as pd
print("Module import success")

import sys
sys.path.append(r'C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\RESOURCES\\python\\utilities\\bootstrap\\')
print("Path append success")

import pprint
from environment import bootstrap_environment  
pprint.pprint(bootstrap_environment(preferred="edmund_gable"))
print("success")

from utilities.bootstrap.environment import bootstrap_environment
env = bootstrap_environment(preferred="edmund_gable")
env
print("Bootstrap environment success")

from utilities import get_dates, get_source_file_dates, find_text
find_text("stats")
print("Utilities import success")

import matplotlib.pyplot as plt
import requests

from pathlib import Path

root = Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026")
raw_data=Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Raw_Data")
processed_data=Path(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Processed_Data")

c:\Users\edmund.gable\.conda\envs\satprocessing\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Module import success
Path append success
{'dataset_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/DATASETS'),
 'hostname': 'NECL04363598',
 'lookuptable_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/lookuptables'),
 'metadata_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources/metadata'),
 'project_root': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'python_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python'),
 'satlogs_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/logs/satprocessing'),
 'timestamp': datetime.datetime(2026, 6, 5, 15, 46, 42, 192186),
 'utilities_path': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/python/utilities/src/utilities'),
 'workflow_resources': WindowsPath('C:/Users/edmund.gable/Documents/Hollings_2026/RESOURCES/workflow_resources')

In [30]:
dsf=xr.open_dataset(r"C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\GitHub\\MOM6-COBALT_2026\\data\\Raw_Data\\Fishbot\\ondeckdata_erddap6.nc")
#find_low_oxygen(SBU01, threshold=5, hypoxia_threshold=2.41, anoxia_threshold=0.2)
dff=dsf.to_dataframe()
print(dff.columns.to_list())

['time', 'temperature', 'dissolved_oxygen', 'dissolved_oxygen_min', 'dissolved_oxygen_max', 'dissolved_oxygen_std', 'salinity', 'data_provider', 'grid_id', 'stat_area', 'fishery_dependent', 'depth', 'latitude', 'longitude']


In [31]:
#Helper function that opens a DO dataset and drops any rows with missing DO values
def open_DO_dataset(file_path, DO_column_name):
    ds=xr.open_dataset(file_path)
    df=ds.to_dataframe()
    df=df.reset_index()
    df=df.dropna(subset=[DO_column_name])
    return df
open_DO_dataset(r"C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\GitHub\\MOM6-COBALT_2026\\data\\Raw_Data\\Fishbot\\ondeckdata_erddap6.nc", 'dissolved_oxygen')
dff=df

In [33]:
#1. Define the DO_column_name variable to match the name of the DO column in your dataset
#ex: DO_column_name="oxygen"
# 2. Open the NetCDF dataset and define it as "df"
dsf=xr.open_dataset(r"C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\GitHub\\MOM6-COBALT_2026\\data\\Raw_Data\\Fishbot\\ondeckdata_erddap6.nc")
dff=dsf.to_dataframe()
print(dff.columns.to_list())
df=open_DO_dataset(r"C:\\Users\\edmund.gable\\Documents\\Hollings_2026\\GitHub\\MOM6-COBALT_2026\\data\\Raw_Data\\Fishbot\\ondeckdata_erddap6.nc", "dissolved_oxygen")
dff=df
# 3. Define the labeling function
def find_low_oxygen(df, DO_column_name, threshold=5, mild_hypoxia_threshold=3.9, hypoxia_threshold=2.41, anoxia_threshold=0.2):
    # Isolate all rows below the upper threshold
    combined_df = df[df[DO_column_name] < threshold].copy()
    
    # Label 'Low Oxygen'
    combined_df['oxygen_status'] = 'Low Oxygen'
    
    # Label 'Mild Hypoxic' (between 3.9 and 2.41)
    combined_df.loc[(combined_df[DO_column_name] < mild_hypoxia_threshold) & (combined_df[DO_column_name] >= hypoxia_threshold), 'oxygen_status'] = 'Mild Hypoxic'
    
    # Label 'Hypoxia' (below 2.41)
    combined_df.loc[combined_df[DO_column_name] < hypoxia_threshold, 'oxygen_status'] = 'Hypoxia'

    # Label 'Anoxia' (below 0.2)
    combined_df.loc[combined_df[DO_column_name] < anoxia_threshold, 'oxygen_status'] = 'Anoxia'

    return combined_df


# 4. Run the function on your DataFrame
Fishbot_hypoxia = find_low_oxygen(dff, 'dissolved_oxygen')

# 5. Print the columns to verify 'depth' is finally there
print(Fishbot_hypoxia.columns.tolist())
Fishbot_hypoxia
Fishbot_hypoxia.to_excel(root / "data" / "Processed_Data" / "Fishbot_Hypoxia.xlsx", index=False)

['time', 'temperature', 'dissolved_oxygen', 'dissolved_oxygen_min', 'dissolved_oxygen_max', 'dissolved_oxygen_std', 'salinity', 'data_provider', 'grid_id', 'stat_area', 'fishery_dependent', 'depth', 'latitude', 'longitude']
['row', 'time', 'temperature', 'dissolved_oxygen', 'dissolved_oxygen_min', 'dissolved_oxygen_max', 'dissolved_oxygen_std', 'salinity', 'data_provider', 'grid_id', 'stat_area', 'fishery_dependent', 'depth', 'latitude', 'longitude', 'oxygen_status']


In [34]:
#Helper Function to drop any data that has NaN values in the DO column
def drop_na_do(df):
    return df.dropna(subset=[DO_column_name])

In [37]:
DO_column_name="DO"
df=open_DO_dataset(root / "data" / "Raw_Data" / "eMOLT" / "eMOLT_2000_2026.nc", "DO")
eMOLT_hypoxia=find_low_oxygen(df, "DO", threshold=5, mild_hypoxia_threshold=3.9, hypoxia_threshold=2.41, anoxia_threshold=0.2)
eMOLT_hypoxia.to_excel(r"C:\Users\edmund.gable\Documents\Hollings_2026\GitHub\MOM6-COBALT_2026\data\Processed_Data\eMOLT_Hypoxia.xlsx", index=False)

In [92]:
ds=xr.open_dataset(root / "data" / "Raw_Data" / "eMOLT" / "eMOLT_2000_2026.nc")
df=ds.to_dataframe()
print(df["time"].min())
df=df.reset_index()
DO_column_name="DO"
df=drop_na_do(df)
eMOLT=find_low_oxygen(df, threshold=5, mild_hypoxia_threshold=3.9, hypoxia_threshold=2.41, anoxia_threshold=0.2)
#eMOLT.to_excel(root / "data" / "Processed_Data" / "eMOLT_Hypoxia.xlsx", index=False)


2023-07-11 17:32:44


In [18]:
ds=xr.open_dataset(root / raw_data / "Hydrographic_Monitoring" / "errdap_search.nc")
dfh=ds.to_dataframe()
print(dfh.columns.to_list())
dfh
unique_cruise=dfh['cruise_id'].dropna().unique()
print(unique_cruise)
unique_cruise_df=pd.DataFrame([{'Cruise Names':unique_cruise}])
unique_cruise_df.to_excel(root / processed_data / 'hydrographic_monitoring_cruises.xlsx', index=False)




['UTC_DATETIME', 'latitude', 'longitude', 'depth', 'dissolved_oxygen', 'cruise_id', 'purpose_code', 'bottom_depth', 'GEAR_TYPE']
<ArrowStringArray>
['DEL0909', 'DEL0911', 'DEL1001', 'DEL1004', 'DEL1012', 'DEL1105', 'DEL1202',
  'HB1202',  'HB1205',  'HB1206',  'PC1206',  'PC1207',  'PC1301',  'GU1302',
  'HB1303',  'HB1304',  'EX1305',  'GU1305',  'GU1401',  'HB1401',  'HB1403',
  'HB1405',  'PC1405',  'HB1501',  'HB1502',  'HB1503',  'GU1506',  'HB1506',
  'HB1601',  'HB1603',  'HB1604',  'PC1607',  'GU1608',  'PC1609',  'GU1701',
  'HB1701',  'HB1702',  'GU1706',  'S11801',  'HB1802',  'S11802',  'HB1803',
  'GU1804',  'HB1805',  'HB1806',  'HB1901',  'S11901',  'GU1902',  'HB1902',
  'GU1905',  'HB1906',  'HB1907',  'HB2101',  'S12101',  'GU2102',  'HB2103',
  'PC2104',  'PC2106',  'S12201',  'HB2202',  'HB2204',  'PC2205',  'HB2207',
  'HB2301',  'S12301',  'HB2302',  'HB2303',  'HB2304',  'PC2305',  'HB2401',
  'HB2402',  'HB2403',  'HB2406',  'PC2406',  'HB2407',  'HB2501',  'HB2

In [20]:


# 1. Reset index and get unique values, then force them into a standard list
cruise_list = dfh['cruise_id'].dropna().unique().tolist()

# 2. Put them in the DataFrame (Make sure there are NO square brackets around cruise_list!)
df_export = pd.DataFrame({'Cruise Names': cruise_list})

# 3. Export to Excel
df_export.to_excel(root / processed_data / 'hydrographic_monitoring_cruises.xlsx', index=False)